# 02 · Reproducible experiments with MLflow

**Goal:** close the MLOps gap. Log parameters, metrics and models for every run, compare runs, reload the
best model. **Time:** ~1.5 hours.

Reproducibility checklist: fixed seeds · pinned package versions (`requirements.txt`) · data version or hash ·
code version (git commit) · every run logged · the evaluation split never changes.

In [1]:
import hashlib, subprocess, warnings
import numpy as np, pandas as pd
import mlflow, mlflow.sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

mlflow.set_tracking_uri("sqlite:///mlflow.db")     # local file DB; view with: mlflow ui --backend-store-uri sqlite:///mlflow.db
mlflow.set_experiment("breast-cancer-baselines")
SEED = 42

In [2]:
X, y = load_breast_cancer(return_X_y=True)
data_hash = hashlib.md5(X.tobytes() + y.tobytes()).hexdigest()[:10]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
try:
    commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
except Exception:
    commit = "no-git"
print("data hash:", data_hash, "| commit:", commit)

data hash: 6c1d3e1d72 | commit: no-git


In [3]:
candidates = {
    "logreg_C1": make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000)),
    "logreg_C0.1": make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=2000)),
    "rf_300": RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
    "histgb": HistGradientBoostingClassifier(random_state=SEED),
}
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
for name, model in candidates.items():
    with mlflow.start_run(run_name=name):
        mlflow.set_tags({"data_hash": data_hash, "git_commit": commit})
        mlflow.log_params({"model": name, "seed": SEED, **{k: v for k, v in model.get_params().items()
                                                          if isinstance(v, (int, float, str, bool)) and len(str(v)) < 100}})
        s = cross_val_score(model, Xtr, ytr, cv=cv, scoring="roc_auc")
        model.fit(Xtr, ytr)
        mlflow.log_metrics({"cv_auc_mean": s.mean(), "cv_auc_std": s.std(),
                            "test_auc": roc_auc_score(yte, model.predict_proba(Xte)[:, 1])})
        # cloudpickle: fine for models you trained yourself. Never unpickle files from untrusted sources;
        # MLflow's default "skops" format exists for that and makes you whitelist tree types explicitly.
        mlflow.sklearn.log_model(model, name="model", serialization_format="cloudpickle")
print("logged", len(candidates), "runs")

logged 4 runs


In [4]:
runs = mlflow.search_runs(order_by=["metrics.cv_auc_mean DESC"])
runs[["tags.mlflow.runName", "metrics.cv_auc_mean", "metrics.cv_auc_std", "metrics.test_auc"]].round(4)

,tags.mlflow.runName,metrics.cv_auc_mean,metrics.cv_auc_std,metrics.test_auc
0,logreg_C1,0.9959,0.0050,0.9954
1,logreg_C0.1,0.9951,0.0046,0.9957
2,histgb,0.9933,0.0034,0.9931
3,rf_300,0.9904,0.0072,0.9937


Choose the model by **CV score**, not test score (otherwise the test set becomes a tuning set).

In [5]:
best_run = runs.iloc[0]
logged = mlflow.search_logged_models(filter_string=f"source_run_id = '{best_run.run_id}'", output_format="list")
best_model = mlflow.sklearn.load_model(logged[0].model_uri)
print("reloaded", best_run["tags.mlflow.runName"], "-> test AUC", round(roc_auc_score(yte, best_model.predict_proba(Xte)[:, 1]), 4))

reloaded logreg_C1 -> test AUC 0.9954


## What else "MLOps" means in an interview
| Stage | Tools to name |
|---|---|
| Experiment tracking | MLflow, Weights & Biases |
| Data / model versioning | DVC, MLflow Model Registry, Hugging Face Hub |
| Pipelines / orchestration | Airflow, Prefect, Kubeflow |
| Serving | FastAPI + Docker, BentoML, vLLM / TGI for LLMs, Triton |
| CI/CD | GitHub Actions: lint, tests, build image, deploy |
| Monitoring | latency/errors (Prometheus-style metrics), data drift (Evidently), model quality on labelled samples |

## ✍️ Your turn
1. Run `mlflow ui --backend-store-uri sqlite:///mlflow.db` in this folder and open http://127.0.0.1:5000.
2. Add MLflow logging to the heart-disease pipeline from module 02.
3. Register the best model in the Model Registry (`mlflow.register_model`).

## 🎯 Interview check
- How do you make an ML experiment reproducible?
- What is data drift vs concept drift? How would you detect each?
- How would you deploy a scikit-learn model behind an API and roll back a bad version?

In [6]:
# Your code here